In [2]:
import sys
sys.path.append('../')

import os
os.environ['CUDA_VISIBLE_DEVICES']='1,2,5,7'

import math

from rich import print

import diffusion_gosai_update
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf
import dataloader_gosai
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime
import oracle
from grelu.lightning import LightningModel
from scipy.stats import pearsonr
import torch
import torch.nn.functional as F
from tqdm import tqdm
import diffusion_gosai_cfg
from utils import set_seed, get_metadata, save_metadata_json
set_seed(0, use_cuda=True)
plt.rcParams['figure.dpi'] = 100

%load_ext autoreload
%autoreload 2

/home/zo122/CHINMAY/papers_with_code/DRAKES/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=> Seed of the run set to 0


In [3]:
BASE_PATH = '/home/zo122/CHINMAY/papers_with_code/DRAKES/data_and_model'
# pretrained model
CKPT_PATH = os.path.join(BASE_PATH, 'mdlm/outputs_gosai/pretrained.ckpt')

# reinitialize Hydra
GlobalHydra.instance().clear()

# Initialize Hydra and compose the configuration|
initialize(config_path="../configs_gosai", job_name="load_model")
cfg = compose(config_name="config_gosai.yaml")
cfg.eval.checkpoint_path = CKPT_PATH

/tmp/ipykernel_600105/227610808.py:9: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../configs_gosai", job_name="load_model")


In [4]:
p_ref = diffusion_gosai_update.Diffusion.load_from_checkpoint(CKPT_PATH, config=cfg)
p_ref.eval()

Diffusion(
  (backbone): CNNModel(
    (linear): Conv1d(5, 128, kernel_size=(9,), stride=(1,), padding=(4,))
    (time_embedder): Sequential(
      (0): GaussianFourierProjection()
      (1): Linear(in_features=128, out_features=128, bias=True)
    )
    (convs): ModuleList(
      (0-7): 8 x Conv1d(128, 128, kernel_size=(9,), stride=(1,), padding=(4,))
      (8-11): 4 x Conv1d(128, 128, kernel_size=(9,), stride=(1,), padding=(16,), dilation=(4,))
      (12-15): 4 x Conv1d(128, 128, kernel_size=(9,), stride=(1,), padding=(64,), dilation=(16,))
      (16-19): 4 x Conv1d(128, 128, kernel_size=(9,), stride=(1,), padding=(256,), dilation=(64,))
    )
    (time_layers): ModuleList(
      (0-19): 20 x Dense(
        (dense): Linear(in_features=128, out_features=128, bias=True)
      )
    )
    (norms): ModuleList(
      (0-19): 20 x LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (final_conv): Sequential(
      (0): Conv1d(128, 128, kernel_size=(1,), stride=(1,))
      (1): R

Define the reward function

In [5]:
reward_model_ft = oracle.get_gosai_oracle(mode='train')
reward_model_ft.eval()

# This is what DRAKES has used - 0.001 (Appendix F.2)
kl_weight = 0.05

@torch.no_grad()
def compute_rewards(seqs) -> torch.Tensor:
    """
    seqs: list of sequences (detokenized ACGT...)
    """
    preds = oracle.cal_gosai_pred_new(seqs, reward_model_ft)
    return torch.tensor(preds[:, 0], device=reward_model_ft.device)

def compute_rewards_with_kl_weight(*args, **kwargs):
    rewards = compute_rewards(*args, **kwargs)
    return rewards / kl_weight

@torch.no_grad()
def compute_rewards_fast(tokens) -> torch.Tensor:
    """
    takes integer tokens directly
    """
    onehot_tokens = F.one_hot(tokens, num_classes=4).float()
    preds = reward_model_ft(onehot_tokens.float().transpose(1, 2)).squeeze()
    return preds[:, 0]

def compute_rewards_with_kl_weight_fast(*args, **kwargs):
    rewards = compute_rewards_fast(*args, **kwargs)
    return rewards / kl_weight

wandb: Currently logged in as: jzinou to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
@torch.no_grad()
def estimate_reward(probs, num_samples, method='mean'):
    B = probs.shape[0]
    dist = torch.distributions.Categorical(probs=probs)
    samples = dist.sample((num_samples,)).reshape(num_samples * B, -1)
    detokenized_samples = dataloader_gosai.batch_dna_detokenize(samples.detach().cpu().numpy())
    rewards = compute_rewards_with_kl_weight(detokenized_samples).reshape(num_samples, B)
    if method == 'mean':
        return rewards.mean(dim=0) # E[r(x)/alpha]
    elif method == 'logmeanexp':
        return rewards.logsumexp(dim=0) - math.log(num_samples) # log E[exp(r(x)/alpha)]
    else:
        raise ValueError(f"Unknown method: {method}")
    
@torch.no_grad()
def estimate_reward_fast(probs, num_samples, method='mean'):
    B = probs.shape[0]
    dist = torch.distributions.Categorical(probs=probs)
    samples = dist.sample((num_samples,)).reshape(num_samples * B, -1)
    rewards = compute_rewards_with_kl_weight_fast(samples).reshape(num_samples, B)
    if method == 'mean':
        return rewards.mean(dim=0) # E[r(x)/alpha]
    elif method == 'logmeanexp':
        return rewards.logsumexp(dim=0) - math.log(num_samples) # log E[exp(r(x)/alpha)]
    else:
        raise ValueError(f"Unknown method: {method}")

In [7]:
# Replace methods with fast methods
compute_rewards = compute_rewards_fast
compute_rewards_with_kl_weight = compute_rewards_with_kl_weight_fast
estimate_reward = estimate_reward_fast

In [8]:
atac_acc_model = LightningModel.load_from_checkpoint(os.path.join(BASE_PATH, 'mdlm/gosai_data/binary_atac_cell_lines.ckpt'), map_location='cuda')
atac_acc_model.eval()

def cal_atac_acc_fast(tokens):
    """
    tokens: list of sequences (tokenized)
    """
    onehot_tokens = F.one_hot(tokens, num_classes=4).float()
    preds = atac_acc_model(onehot_tokens.float().transpose(1, 2)).detach().cpu().numpy()
    preds = preds.squeeze() # numpy array with shape [n_seqs, 7]
    return (preds[:,1]>0.5).sum()/len(preds)

Initialize model to train

In [9]:
q_phi = diffusion_gosai_update.Diffusion.load_from_checkpoint(CKPT_PATH, config=cfg)
q_phi.eval()
num_timesteps = q_phi.config.sampling.steps
f_psi = torch.nn.Parameter(torch.zeros(num_timesteps, device=q_phi.device))

In [10]:
q_phi.load_state_dict(torch.load(os.path.join(BASE_PATH, 'mdlm/reward_bp_results_final/finetuned.ckpt')))

<All keys matched successfully>

In [ ]:
batch_size = 64
lr = 0.0001
optimizer = torch.optim.Adam(list(q_phi.parameters()) + [f_psi], lr=lr)
num_epochs = 1
batches_per_epoch = 10
patience = 3
sample_onpolicy = True
num_samples_for_reward_estimate = 20
reward_estimate_method = 'logmeanexp'
timesteps_for_loss = 16
regularization_strength = 1.0

base_dir = 'model_weights'  # keep base folder
timestamp = datetime.now().strftime("%Y%m%d/%H%M%S")  # e.g. 20250818/004927
model_save_dir = os.path.join(base_dir, timestamp)

os.makedirs(model_save_dir, exist_ok=True)
ckpt_path_best_loss = f'{model_save_dir}/best_loss.pth'
ckpt_path_best_reward = f'{model_save_dir}/best_reward.pth'
ckpt_path_best_atac_acc = f'{model_save_dir}/best_atac_acc.pth'

In [12]:
# Save config and metadata files

OmegaConf.save(config=cfg, f=f'{model_save_dir}/config.yaml')

metadata = get_metadata(dict(locals()), ignore_hidden=True)
print(metadata)
save_metadata_json(metadata, model_save_dir)

{
    'BASE_PATH': '/home/zo122/CHINMAY/papers_with_code/DRAKES/data_and_model',
    'CKPT_PATH': '/home/zo122/CHINMAY/papers_with_code/DRAKES/data_and_model/mdlm/outputs_gosai/pretrained.ckpt',
    'kl_weight': 0.05,
    'num_timesteps': 128,
    'batch_size': 64,
    'lr': 0.0001,
    'num_epochs': 1,
    'batches_per_epoch': 10,
    'patience': 3,
    'sample_onpolicy': True,
    'num_samples_for_reward_estimate': 20,
    'reward_estimate_method': 'logmeanexp',
    'timesteps_for_loss': 16,
    'regularization_strength': 1.0,
    'base_dir': 'model_weights',
    'timestamp': '20250821/214612',
    'model_save_dir': 'model_weights/20250821/214612',
    'ckpt_path_best_loss': 'model_weights/20250821/214612/best_loss.pth',
    'ckpt_path_best_reward': 'model_weights/20250821/214612/best_reward.pth'
}

Metadata saved to model_weights/20250821/214612/metadata.json


In [ ]:
loss_trace = []
reward_trace = []
atac_acc_trace = []

In [12]:
# torch.autograd.set_detect_anomaly(True)

In [19]:
L = q_phi.config.model.length
eps=1e-5
timesteps = torch.linspace(1, eps, num_timesteps + 1, device=q_phi.device)
dt = (1 - eps) / num_timesteps

# Training loop
for epoch in range(num_epochs):
    total_epoch_loss = 0.0
    total_epoch_rewards = 0.0
    total_epoch_atac_acc = 0.0
    for batch_idx in range(batches_per_epoch):
        q_phi.train()
        
        rewards_prev = None
        log_prob_p_ref = None
        log_prob_q_phi = None
        loss = torch.tensor(0.0, device=q_phi.device)
        kld_regularization = torch.tensor(0.0, device=q_phi.device)
        
        # We select only #timesteps_for_loss timesteps randomly for loss calculation to fit in memory
        is_selected_timestep = torch.zeros(
            num_timesteps, dtype=torch.bool
        ).scatter_(0, torch.randperm(num_timesteps)[:timesteps_for_loss], True)
        
        # Generate batch_size samples from q_phi
        z_t = q_phi._sample_prior(batch_size, L).to(q_phi.device)
        for i in range(num_timesteps):
            t = timesteps[i] * torch.ones(z_t.shape[0], 1, device=q_phi.device)
            # Invoke pretrained and finetune models
            with torch.enable_grad() if is_selected_timestep[i-1] else torch.no_grad():
                q_phi_zs_given_zt, q_phi_z0_given_zt = q_phi._sample_step(z_t, t, dt)
            with torch.no_grad():
                p_ref_zs_given_zt, p_ref_z0_given_zt = p_ref._sample_step(z_t, t, dt)
                
            if is_selected_timestep[i-1]:
                kld_batch = torch.where(
                    p_ref_z0_given_zt > 0,
                    p_ref_z0_given_zt * (torch.log(p_ref_z0_given_zt) - torch.log(q_phi_z0_given_zt.clamp_min(1e-12))),
                    torch.zeros_like(p_ref_z0_given_zt)
                ).sum(dim=(1, 2))
                kld_regularization += kld_batch.mean(dim=0) # take mean across batch dimension
                
            # Estimate rewards
            rewards = estimate_reward(p_ref_z0_given_zt, num_samples_for_reward_estimate, method=reward_estimate_method)
            
            if rewards_prev is not None and is_selected_timestep[i]:
                assert log_prob_p_ref is not None and log_prob_q_phi is not None
                
                log_w = (rewards - rewards_prev) + (log_prob_p_ref - log_prob_q_phi) # Shape: (batch-size,)
                log_variance = (log_w - f_psi[i]) ** 2
                loss += log_variance.mean(dim=0) # take mean across batch dimension
            
            q_phi_dist = torch.distributions.Categorical(probs=q_phi_zs_given_zt)
            p_ref_dist = torch.distributions.Categorical(probs=p_ref_zs_given_zt)
            
            if sample_onpolicy:
                z_s = q_phi_dist.sample()
            else:
                z_s = p_ref_dist.sample()
                
            log_prob_q_phi = q_phi_dist.log_prob(z_s).sum(dim=1)
            log_prob_p_ref = p_ref_dist.log_prob(z_s).sum(dim=1)
            
            # Update for next step
            z_t = z_s
            rewards_prev = rewards
            
        z_0 = z_t
        if q_phi.config.sampling.noise_removal:
            with torch.no_grad():
                t = timesteps[-1] * torch.ones(z_0.shape[0], 1, device=q_phi.device)
                unet_conditioning = q_phi.noise(t)[0]
                logits = q_phi.forward(z_0, unet_conditioning)
                z_0 = logits[:, :, :-1].argmax(dim=-1)
        
        # Compute rewards
        rewards = compute_rewards_with_kl_weight(z_0)
        total_epoch_rewards += rewards.sum(dim=0).item() * kl_weight # because the rewards we have is with kl weight r(x)/kl_weight
        
        if is_selected_timestep[0]:
            assert rewards_prev is not None and log_prob_p_ref is not None and log_prob_q_phi is not None
            log_w = (rewards - rewards_prev) + (log_prob_p_ref - log_prob_q_phi) # Shape: (batch-size,)
            log_variance = (log_w - f_psi[0]) ** 2
            loss += log_variance.mean(dim=0) # take mean across batch dimension
        
        # Add KL regularization
        loss += regularization_strength * kld_regularization
        
        # Backpropagation
        optimizer.zero_grad()
        # loss.backward()
        # optimizer.step()
        
        atac_acc = cal_atac_acc_fast(z_0)
        total_epoch_atac_acc += atac_acc.item()

        total_epoch_loss += loss.item()
        print((f"Batch {batch_idx+1}/{batches_per_epoch}, "
               f"Loss: {loss.item()}, Reward (avg): {rewards.mean(dim=0).item() * kl_weight} "
               f"KL Regularization: {kld_regularization.item() * regularization_strength} "
               f"ATAC Accuracy: {atac_acc.item()}"))
    
    q_phi.eval()
    avg_loss = total_epoch_loss / batches_per_epoch
    avg_rewards = total_epoch_rewards / (batches_per_epoch * batch_size)
    avg_atac_acc = total_epoch_atac_acc / batches_per_epoch
    
    print(f"Epoch {epoch+1}/{num_epochs},  Loss (avg): {avg_loss}, Reward (avg): {avg_rewards}, ATAC acc (avg): {avg_atac_acc}")
    loss_trace.append(avg_loss)
    reward_trace.append(avg_rewards)
    atac_acc_trace.append(avg_atac_acc)
    
    if loss_trace[-1] == min(loss_trace):
        # store model weights
        torch.save(q_phi.state_dict(), ckpt_path_best_loss)
        print(f"Best loss yet! Saved model weights to {ckpt_path_best_loss}")
    if reward_trace[-1] == max(reward_trace):
        # store model weights
        torch.save(q_phi.state_dict(), ckpt_path_best_reward)
        print(f"Best reward yet! Saved model weights to {ckpt_path_best_reward}")
    if atac_acc_trace[-1] == max(atac_acc_trace):
        # store model weights
        torch.save(q_phi.state_dict(), ckpt_path_best_atac_acc)
        print(f"Best ATAC accuracy yet! Saved model weights to {ckpt_path_best_atac_acc}")
        
    # If BOTH loss and reward stop imporving, then stop training
    if (
        min(loss_trace) < min(loss_trace[-patience:]) and 
        max(reward_trace) > max(reward_trace[-patience:]) and 
        max(atac_acc_trace) > max(atac_acc_trace[-patience:])
    ):
        break

Batch 1/10, Loss: 8348.134765625, Reward (avg): 5.691080474853516 KL Regularization: 170.23260498046875 ATAC 
Accuracy: 0.9375

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# loss_trace and reward_trace are 1D lists (or 1D arrays) of the same length
epochs = range(1, len(loss_trace) + 1)

fig, ax1 = plt.subplots()
ax1.plot(epochs, loss_trace, label='Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2 = ax1.twinx()
ax2.plot(epochs, reward_trace, label='Reward', color='green')
ax2.set_ylabel('Reward')

# place legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Loss and Reward vs. Epoch')
plt.show()

Load best loss model

In [ ]:
# q_phi.load_state_dict(torch.load(ckpt_path_best_loss))

Load best reward model

In [ ]:
# q_phi.load_state_dict(torch.load(ckpt_path_best_reward))